<a href="https://colab.research.google.com/github/mounika6776/Alarm-manager-with-Notification/blob/master/Day2_setup_resume_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import os, getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [ ]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [ ]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

        except ValidationError as e:

            if attempt == max_retries:
                raise

            fix_prompt = (
                f'Fix this JSON to match schema. Errors: {e}. '
                f'Original: {resp.text}'
            )

            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

In [20]:
# Load sample résumés from the lab kit
with open('./sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e)._name_}: {str(e)[:200]}')

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 5 sample résumés

Résumé 1: Ravi Kumar — 6 skills, 0.25 years exp

Résumé 2: Sneha Reddy — 6 skills, 0.115 years exp

Résumé 3: Arun Pillai — 8 skills, 0.5 years exp

=== Full first result ===
{
  "name": "Ravi Kumar",
  "email": "ravi.kumar@example.com",
  "phone": "+91-9876543210",
  "education": [
    {
      "degree": "B.Tech Computer Science",
      "institution": "Aditya University",
      "year": 2026
    },
    {
      "degree": "Intermediate",
      "institution": "Narayana Junior College",
      "year": 2022
    }
  ],
  "skills": [
    "Python",
    "Java",
    "SQL",
    "Git",
    "Linux",
    "REST APIs"
  ],
  "projects": [
    "Built a Flask REST API for college placement portal",
    "Solved 250+ DSA problems on LeetCode",
    "Final-year project: ML model for crop yield prediction"
  ],
  "experience_years": 0.25
}


In [1]:
# Empty string — should fail gracefully, not crash
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Caught gracefully: NameError
Message: name 'extract_resume' is not defined
